# Entrega 3 — Chunking de Manifestações Longas

Usa `RecursiveCharacterTextSplitter` e compara duas configurações de tamanho e overlap.

In [ ]:
from pathlib import Path
import json,pandas as pd
import matplotlib.pyplot as plt
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

ROOT=Path.cwd()
if not (ROOT/'data'/'manifestacoes.json').exists(): ROOT=ROOT.parent
with open(ROOT/'data'/'manifestacoes.json',encoding='utf-8') as f: data=json.load(f)
df=pd.DataFrame(data)
longas=df.assign(nchars=df.texto.str.len()).sort_values("nchars",ascending=False).head(5)
longas[["id","nchars","texto"]]


In [ ]:
def gerar_chunks(texto,size,overlap):
    splitter=RecursiveCharacterTextSplitter(
        chunk_size=size,chunk_overlap=overlap,
        separators=["\n\n","\n",". "," ",""]
    )
    return splitter.split_text(texto)

configs=[(250,30),(400,80)]
for size,overlap in configs:
    print(f"\nCONFIGURAÇÃO size={size}, overlap={overlap}")
    for _,r in longas.iterrows():
        chunks=gerar_chunks(r.texto,size,overlap)
        print(r.id,len(chunks),[len(c) for c in chunks])


In [ ]:
model=SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
rows=[]
for _,r in longas.iterrows():
    chunks=gerar_chunks(r.texto,400,80)
    for k,c in enumerate(chunks):
        rows.append({"id":r.id,"chunk":k+1,"texto":c})
cdf=pd.DataFrame(rows)
E=model.encode(cdf.texto.tolist(),normalize_embeddings=True,show_progress_bar=False)
xy=PCA(n_components=2,random_state=42).fit_transform(E)
cdf["x"]=xy[:,0]; cdf["y"]=xy[:,1]

for ident,g in cdf.groupby("id"):
    plt.figure(figsize=(7,5))
    plt.scatter(g.x,g.y)
    for _,r in g.iterrows(): plt.annotate(str(r.chunk),(r.x,r.y))
    plt.title(f"Chunks de {ident} — PCA")
    plt.show()


## Discussão

O overlap reaproveita parte do conteúdo entre chunks consecutivos. Isso tende a reduzir a perda de contexto nas fronteiras, pois uma informação que atravessa o limite pode aparecer em dois chunks. Overlap muito pequeno pode separar contexto; overlap excessivo aumenta redundância e custo computacional. A escolha final deve considerar os gráficos e exemplos produzidos para as cinco manifestações mais longas do dataset oficial.